# Day 4

## Tokenizing with code

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
tokens = encoding.encode("My Name is Pratham K and I am also called The Great")

In [2]:
tokens

[5444, 7317, 382, 2284, 94125, 658, 326, 357, 939, 1217, 4358, 623, 10405]

In [3]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

5444 = My
7317 =  Name
382 =  is
2284 =  Pr
94125 = atham
658 =  K
326 =  and
357 =  I
939 =  am
1217 =  also
4358 =  called
623 =  The
10405 =  Great


In [4]:
encoding.decode([7317])

' Name'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [6]:
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [17]:
from openai import OpenAI

OLLAMA_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_URL,api_key="ollama")

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Pratham K!"}
    ]

In [20]:
response = ollama.chat.completions.create(model="llama3.2:latest", messages=messages)
response.choices[0].message.content

"Hello Pratham K! It's nice to meet you. Is there something I can help you with or would you like to chat?"

### OK let's now ask a follow-up question

In [21]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [22]:
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

"I don't know your name. You didn't tell me during our conversation so far. Would you like to share it with me? I can address you as you would like."

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [25]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Pratham K!"},
    {"role": "assistant", "content": "Hi Pratham K! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [26]:
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

"Your name is Pratham K. Is there something specific you'd like to know about your name or anything else? I'm here to help!"

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

